In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark import SparkConf

conf = SparkConf().setAppName("dmltest")
conf.set('spark.jars.packages', 'io.delta:delta-core_2.12:2.1.0')
conf.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
conf.set("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore")

In [2]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()

your 131072x1 screen size is bogus. expect trouble
25/04/18 04:38:04 WARN Utils: Your hostname, xRhl resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/04/18 04:38:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/rahul/micromamba/envs/wsl-pyspark/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/rahul/.ivy2/cache
The jars for the packages stored in: /home/rahul/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-77d4cab3-1b9e-4177-933a-880b86277fb3;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.1.0 in central
	found io.delta#delta-storage;2.1.0 in central
	found org.antlr#antlr4-runtime;4.8 in central
	found org.codehaus.jackson#jackson-core-asl;1.9.13 in central
:: resolution report :: resolve 389ms :: artifacts dl 28ms
	:: modules in use:
	io.delta#delta-core_2.12;2.1.0 from central in [default]
	io.delta#delta-storage;2.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.8 from central in [default]
	org.codehaus.jackson#jackson-core-asl;1.9.13 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evic

In [ ]:
df_trp = spark.read.format('parquet').load('/home/glue_user/workspace/sparklearning/src_data_pq/tripdata')
df_cus = spark.read.format('csv').options(header='true', inferschema='true').load('/home/glue_user/workspace/sparklearning/src_data_csv/Customer.csv')
df_ren = spark.read.format('csv').options(header='true', inferschema='true').load('/home/glue_user/workspace/sparklearning/src_data_csv/dvdrental.csv')
df_stf = spark.read.format('csv').options(header='true', inferschema='true').load('/home/glue_user/workspace/sparklearning/src_data_csv/Staff.csv')


In [ ]:
#selectExpr
df_cus.selectExpr("*", "Company_name nameOfCompany_using_selectExpr").limit(2).show(truncate=False)
df_cus.select("*", F.col('Company_name').alias('nameOfCompany_using_alias')).limit(2).show(truncate=False)
df_cus.withColumnRenamed('Company_name', 'nameOfCompany_using_withColumnRenamed').limit(2).show(truncate=False)

In [ ]:
#groupBy
df_cus.groupBy('Town').agg(F.avg('Town').alias('zavg'), F.count('Town').alias('cnt')).show()
df_cus.groupBy('Town').agg(F.expr("count(*) cnt"), F.expr("cast(sum(Company_ref) as integer) sm")).show()

In [ ]:
#windowFunction
df_cus.selectExpr("*", 
                  "row_number() over(partition by Town order by Company_ref) as rn",
                  "count(*) over(partition by Town) cnt").filter("cnt>1")\
                    .orderBy(F.expr("town"), F.expr("rn desc")).show()

In [ ]:
spark.sql('''
select
    to_timestamp('1993-08-15T10:30:45.5+05:30') bd
''').show(truncate=False)

In [ ]:
df_ren.printSchema()

In [ ]:
#join
df_ren.createOrReplaceTempView('dvdrental')
df_cus.createOrReplaceTempView('customer')
df_stf.createOrReplaceTempView('staff')

spark.sql('''
select
    *
from
    dvdrental
join
    customer
    on dvdrental.customer_id=customer.customer_id
''').show()

In [ ]:
spark.range(1000).write.mode('overwrite').format('delta').saveAsTable('cloud.test_delta')

In [ ]:
df = spark.read.table("cloud.test_delta")

In [3]:
spark.conf.get("spark.sql.warehouse.dir")

'file:/mnt/c/Users/raulr/OneDrive/Works/learning/de-skill/de-devspace/data-engineering/2025/code/python-spark-sql/02_dml/dml/spark-warehouse'

In [6]:
spark.stop()